# GenAI API 

## Langchain

### 1. Data Ingestion 

In [20]:
# import
## Data Ingestging
import bs4
from langchain_community.document_loaders import TextLoader, PyPDFLoader, WebBaseLoader, ArxivLoader, WikipediaLoader

In [22]:
docs_pdf = PyPDFLoader('attention.pdf').load()
loader = WebBaseLoader(web_paths=("https://lilianweng.github.io/posts/2023-06-23-agent/",),
                     bs_kwargs=dict(parse_only=bs4.SoupStrainer(
                         class_=("post-title","post-content","post-header")
                     )))
docs_ark = ArxivLoader(query="1706.03762", load_max_docs=2).load()
docs_wk = WikipediaLoader(query="Generative AI", load_max_docs=2).load()

text_documents = TextLoader('speech.txt').load()

In [3]:
# text_documents
# docs_pdf
# len(docs_ark)
# print(docs_wk)

### 2. Data Transformation

**Text Splitting from Documents- RecursiveCharacter Text Splitters**

This text splitter is the recommended one for generic text. It is parameterized by a list of characters. It tries to split on them in order until the chunks are small enough. The default list is ["\n\n", "\n", " ", ""]. This has the effect of trying to keep all paragraphs (and then sentences, and then words) together as long as possible, as those would generically seem to be the strongest semantically related pieces of text.

- **How the text is split:** by list of characters.
- **How the chunk size is measured:** by number of characters.

In [4]:
# Import 
## Data transformation/Chunk
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_text_splitters import CharacterTextSplitter
from langchain_text_splitters import HTMLHeaderTextSplitter

In [5]:
# print(final_documents[0])
# print(text[0])

In [23]:

text_splitter = CharacterTextSplitter(separator="\n\n",  chunk_size=100, chunk_overlap=20)
final_documents = text_splitter.split_documents(text_documents)

Created a chunk of size 470, which is longer than the specified 100
Created a chunk of size 347, which is longer than the specified 100
Created a chunk of size 668, which is longer than the specified 100
Created a chunk of size 982, which is longer than the specified 100
Created a chunk of size 789, which is longer than the specified 100


**How to split by HTML header**

**How to split JSON data**

### 3. Embeddings

In [7]:
# Import
import os
from dotenv import load_dotenv

# OpenaAi embedding 
from langchain_openai import OpenAIEmbeddings
from langchain.embeddings import OpenAIEmbeddings

# Ollama embedding
from langchain_community.embeddings import OllamaEmbeddings
# Huggingface embedding

load_dotenv() 
os.environ["OPENAI_API_KEY"]=os.getenv("OPENAI_API_KEY")
os.environ['HF_TOKEN']=os.getenv("HF_TOKEN")

**1. OpenAI Embedding**

[OpenAIEmbedding-Reference](https://platform.openai.com/docs/guides/embeddings)

In [8]:
embeddings_1024=OpenAIEmbeddings(model="text-embedding-3-large", dimensions=1024)

# Example
text="This is a tutorial on OPENAI embedding"
query_result=embeddings_1024.embed_query(text)
#query_result

C:\Users\tazeb\AppData\Local\Temp\ipykernel_51668\190547378.py:1: LangChainDeprecationWarning: The class `OpenAIEmbeddings` was deprecated in LangChain 0.0.9 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-openai package and should be used instead. To use it run `pip install -U :class:`~langchain-openai` and import as `from :class:`~langchain_openai import OpenAIEmbeddings``.
  embeddings_1024=OpenAIEmbeddings(model="text-embedding-3-large", dimensions=1024)
c:\Users\tazeb\OneDrive\AtomicHabit\LLM Engineering\venv\Lib\site-packages\langchain_community\embeddings\openai.py:271: UserWarning: WARNING! dimensions is not default parameter.
                    dimensions was transferred to model_kwargs.
                    Please confirm that dimensions is what you intended.
  warnings.warn(


**2. Ollama Embedding**

[OllamaEmbedding-Reference](https://api.python.langchain.com/en/latest/ollama/embeddings/langchain_ollama.embeddings.OllamaEmbeddings.html)

In [9]:
#!pip install langchain ollama

In [10]:
from langchain.embeddings import OllamaEmbeddings

embeddings_ollama = OllamaEmbeddings(model="llama3:latest")
r1 = [embeddings_ollama.embed_query(text) for text in [
    "Alpha is the first letter of Greek alphabet", 
    "Beta is the second letter of Greek alphabet"
]]
#r1

C:\Users\tazeb\AppData\Local\Temp\ipykernel_51668\4071371012.py:3: LangChainDeprecationWarning: The class `OllamaEmbeddings` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the :class:`~langchain-ollama package and should be used instead. To use it run `pip install -U :class:`~langchain-ollama` and import as `from :class:`~langchain_ollama import OllamaEmbeddings``.
  embeddings_ollama = OllamaEmbeddings(model="llama3:latest")


In [11]:
embeddings_deepseek=(OllamaEmbeddings(model="deepseek-r1:latest"))  
r2=embeddings_deepseek.embed_documents(["Alpha is the first letter of Greek alphabet", "Beta is the second letter of Greek alphabet"])
#r2

**3. Huggingface Embedding**

[huggingFace-Embedding-Reference](https://api.python.langchain.com/en/latest/huggingface/embeddings/langchain_huggingface.embeddings.huggingface.HuggingFaceEmbeddings.html)

In [13]:
from langchain_huggingface import HuggingFaceEmbeddings
embeddings=HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
text="this is atest documents"
query_result=embeddings.embed_query(text)
doc_result = embeddings.embed_documents([text, "This is not a test document."])
#doc_result[0]

c:\Users\tazeb\OneDrive\AtomicHabit\LLM Engineering\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\tazeb\OneDrive\AtomicHabit\LLM Engineering\venv\Lib\site-packages\huggingface_hub\file_download.py:795: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


### 4. VectorStore

[chroma-reference](https://api.python.langchain.com/en/latest/vectorstores/langchain_chroma.vectorstores.Chroma.html)

[faiss-reference](https://python.langchain.com/api_reference/community/vectorstores/langchain_community.vectorstores.faiss.FAISS.html)

In [14]:
# Import
from langchain_chroma import Chroma
from langchain_community.vectorstores import FAISS

**1. FAISS**

In [15]:
# Vectorstore
db = FAISS.from_documents(final_documents, embeddings)

### querying 
query = "How does the speaker describe the desired outcome of the war?"
docs = db.similarity_search(query)
docs[0].page_content

'Just because we fight without rancor and without selfish object, seeking nothing for ourselves but what we shall wish to share with all free peoples, we shall, I feel confident, conduct our operations as belligerents without passion and ourselves observe with proud punctilio the principles of right and of fair play we profess to be fighting for.'

As a Retriever

We can also convert the vectorstore into a Retriever class. This allows us to easily use it in other LangChain methods, which largely work with retrievers

In [16]:
retriever = db.as_retriever()
docs = retriever.invoke(query)
docs[0].page_content

'Just because we fight without rancor and without selfish object, seeking nothing for ourselves but what we shall wish to share with all free peoples, we shall, I feel confident, conduct our operations as belligerents without passion and ourselves observe with proud punctilio the principles of right and of fair play we profess to be fighting for.'

Similarity Search with score

There are some FAISS specific methods. One of them is similarity_search_with_score, which allows you to return not only the documents but also the distance score of the query to them. The returned distance score is L2 distance. Therefore, a lower score is better.

In [17]:
docs_and_score = db.similarity_search_with_score(query)

### Saving And Loading
db.save_local("faiss_index")

new_db = FAISS.load_local("faiss_index", embeddings,allow_dangerous_deserialization = True)
docs = new_db.similarity_search(query)

**2. Chroma**

In [18]:
vectordb=Chroma.from_documents(documents=final_documents,embedding=embeddings,persist_directory="./chroma_db")
# load from disk
db2 = Chroma(persist_directory="./chroma_db", embedding_function=embeddings)
docs=db2.similarity_search(query)
print(docs[0].page_content)

## similarity Search With Score
docs = vectordb.similarity_search_with_score(query)

### Retriever option
retriever=vectordb.as_retriever()
retriever.invoke(query)[0].page_content

Just because we fight without rancor and without selfish object, seeking nothing for ourselves but what we shall wish to share with all free peoples, we shall, I feel confident, conduct our operations as belligerents without passion and ourselves observe with proud punctilio the principles of right and of fair play we profess to be fighting for.


'Just because we fight without rancor and without selfish object, seeking nothing for ourselves but what we shall wish to share with all free peoples, we shall, I feel confident, conduct our operations as belligerents without passion and ourselves observe with proud punctilio the principles of right and of fair play we profess to be fighting for.'